# 07 · When things go wrong

*Arm each failure on purpose, then write code that survives it.*

**About 25 minutes.**

> **Not a medical device. Teaching use only.** Everything here talks to a *simulator* running on
> your own machine. The surfaces are synthetic, the clearance maps are plausible rather than
> physical, and nothing in this repository came from a patient. Not affiliated with or endorsed
> by Vision RT.

First, make sure the environment is ready. On your own machine this cell does nothing at all — it only matters on Colab and Binder, where the simulator has to be installed and started from inside the notebook.

In [ ]:
# Running on Google Colab? This installs the workshop and starts the simulator.
# Anywhere else it does nothing: locally you already started it in the other terminal, and on
# Binder it was started for you before this notebook opened.
import os, subprocess, sys, time, urllib.request

if "COLAB_RELEASE_TAG" in os.environ:
    REPO = "https://github.com/tallhamer/SGRT-Scripting-Workshop"
    CHECKOUT = "/content/workshop"

    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    f"maprt-scripting-workshop[notebooks] @ git+{REPO}"], check=True)

    if not os.path.isdir(CHECKOUT):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, CHECKOUT], check=True)

    # Work from the checkout's notebooks/ directory. Colab opens a notebook detached from its
    # repository, so every relative path would otherwise be wrong — the capstone reads
    # "../data/plans/..." and would look in /data. One chdir makes Colab match a local clone
    # instead of special-casing paths in each notebook.
    os.chdir(f"{CHECKOUT}/notebooks")

    os.environ["MAPRT_SIM_TOKEN"] = "workshop-colab-token"
    os.environ["MAPRT_SIM_URL"] = "http://127.0.0.1:8000"

    # --token matters: without it the simulator mints a random token, while the environment
    # above advertises a different one, and every call fails with InvalidClientToken. The
    # simulator must be given the same secret this notebook is going to send.
    log = open("/content/maprt-sim.log", "w")
    subprocess.Popen(["maprt-sim", "--port", "8000",
                      "--token", os.environ["MAPRT_SIM_TOKEN"],
                      "--data-dir", f"{CHECKOUT}/data",
                      "--map-seconds", "1"],
                     stdout=log, stderr=subprocess.STDOUT)

    # Poll rather than sleep. A fixed wait is either too short on a slow runtime (the first cell
    # fails for no visible reason) or wasted time on a fast one.
    for _attempt in range(60):
        try:
            urllib.request.urlopen(os.environ["MAPRT_SIM_URL"] + "/sim/state", timeout=1)
            print("simulator ready")
            break
        except Exception:
            time.sleep(0.5)
    else:
        print("The simulator did not start. See /content/maprt-sim.log")

print("Python", sys.version.split()[0])

In [ ]:
from maprt_workshop import connect

maprt = connect()          # finds the simulator and your token, and checks it answers
print(maprt.api_url)

## Why this notebook exists

Error handling is the part of a client that never gets tested, because the failures it exists for
are the ones you cannot arrange. An unlicensed appliance needs an appliance without the licence; a
throttled server needs a busy one; a timeout needs patience.

So it gets written once, guessed at, and never exercised — and then it is what fails during
treatment planning on a Friday afternoon.

The simulator can be *told* to fail. Everything below is armed deliberately.

In [ ]:
import httpx2

headers = dict(maprt.headers)
SIM = maprt.api_url + "/sim"


def arm(name):
    """Arm a fault. Returns what it is meant to teach."""
    response = httpx2.post(SIM + "/scenario", json={"name": name})
    response.raise_for_status()
    return response.json()


def disarm():
    return arm("ok")


for scenario in httpx2.get(SIM + "/scenarios").json()["scenarios"]:
    print(f"{scenario['name']:<22} {scenario['description']}")

> `/sim` is **not part of the MapRT API.** It is the simulator's own control surface, and it is
> deliberately unauthenticated — the `bad-token` scenario rejects every token, so if disarming it
> needed a working one you would have to restart the server to get out.

## 1. The licensed features

Surfaces, clearance maps and beam-delivery status are each separately licensed.

In [ ]:
print(arm("unlicensed-map")["teaches"])

In [ ]:
body = {"PatientSurfaceId": maprt.get_patient_surfaces("ANON001")[0][0],
        "TreatmentRoomId": maprt.get_treatment_rooms()[0][1],
        "PatientPosition": "HFS", "CouchBuffer": 20.0, "PatientBuffer": 20.0,
        "HighResolution": False, "Isocenter": {"x": 0.0, "y": 0.0, "z": 0.0}}

response = httpx2.post(maprt.api_url + "/integration/GetMap", headers=headers, json=body)

print("status      :", response.status_code)
print("content-type:", response.headers["Content-Type"])
print("body        :", response.text[:90])

**Look at the content type.** A successful `GetMap` is `text/plain`; this one is JSON. That is why
`parse_map_response` sniffs the body before deciding how to read it — a parser that assumed text
would try to read `{"data": null, ...}` as a clearance map and produce nonsense.

In [ ]:
from maprt_workshop import MapRTApiError, wire

try:
    wire.parse_map_response(response.text)
except MapRTApiError as problem:
    print("raised:", problem)
    print("category:", problem.category)

And the thing to actually do about it: **`license` is not `no`.**

```python
except MapRTApiError as problem:
    if problem.category == "license":
        show("This appliance is not licensed for clearance maps.")   # we cannot answer
    else:
        show("Could not compute a clearance map.")                   # something broke
```

If an unlicensed map renders as "no safe angles found", you have turned *"I don't know"* into
*"the answer is no"*, in a tool whose job is deciding whether a beam can be delivered.

In [ ]:
disarm();

## 2. Timeouts

A real `GetMap` takes **minutes**. The documented client default is **180 seconds** — not five.

In [ ]:
print(arm("slow")["teaches"])

In [ ]:
import time

started = time.monotonic()
try:
    httpx2.get(maprt.api_url + "/integration/ping", headers=headers, timeout=2.0)
except httpx2.TimeoutException as problem:
    print(f"gave up after {time.monotonic() - started:.1f}s: {type(problem).__name__}")

Nothing was wrong with the appliance. Your client decided it was broken.

Note the exception type: a timeout is **not** a `MapRTApiError`. It never reached the appliance,
so there is no envelope and no error code — it is a transport failure, and your handling has to
cover both kinds.

In [ ]:
started = time.monotonic()
response = httpx2.get(maprt.api_url + "/integration/ping", headers=headers, timeout=30.0)
print(f"with a sensible timeout: {response.json()['data']} after "
      f"{time.monotonic() - started:.1f}s")
disarm();

Pick timeouts per endpoint. `ping` should answer instantly and a 10-second wait means something
is wrong. `GetMap` legitimately takes minutes.

In [ ]:
TIMEOUTS = {
    "ping": 10.0,
    "rooms": 30.0,
    "surfaces": 120.0,       # meshes are large
    "GetMap": 600.0,         # "may take up to a few minutes"
}
TIMEOUTS

## 3. Throttling

From §2.2.1.3: the server *"may have throttling mechanisms implemented and report to the client if
it cannot process immediately and the request should be repeated."*

In [ ]:
print(arm("throttle")["teaches"])

In [ ]:
response = httpx2.get(maprt.api_url + "/integration/ping", headers=headers)

print("status     :", response.status_code)
print("Retry-After:", response.headers.get("Retry-After"), "seconds")

**503 with a `Retry-After` header.** The server is telling you exactly how long to wait. Use it
rather than guessing — and *do* back off, because a tight retry loop against a struggling
appliance makes it worse.

Here is a retry that handles it properly. Note what it does **not** retry:

In [ ]:
def call_with_retry(method, endpoint, body=None, attempts=5, timeout=60.0):
    """A MapRT call that survives throttling and transient server trouble."""
    delay = 1.0
    for attempt in range(1, attempts + 1):
        try:
            response = httpx2.request(method, maprt.api_url + endpoint,
                                      headers=headers, json=body, timeout=timeout)
        except httpx2.TimeoutException:
            if attempt == attempts:
                raise
            wait = delay
        else:
            if response.status_code == 503:
                # The server said "later" and told us how long.
                wait = float(response.headers.get("Retry-After", delay))
            elif response.status_code >= 500:
                wait = delay                      # transient: worth another go
            elif response.status_code >= 400:
                # OUR request is wrong. Retrying it unchanged fails identically, faster.
                response.raise_for_status()
            else:
                envelope = response.json()
                if not envelope.get("success", True):
                    # A refusal, not a failure. Retrying changes nothing.
                    raise MapRTApiError(envelope["errorCode"], envelope["errorCodeType"])
                return envelope["data"]

        print(f"  attempt {attempt} failed; waiting {wait:.0f}s")
        time.sleep(wait)
        delay = min(delay * 2, 30.0)              # exponential, capped

    raise RuntimeError(f"{endpoint} failed after {attempts} attempts")


arm("throttle")
print("result:", call_with_retry("GET", "/integration/ping"))

Three categories, three behaviours, and the distinction is the whole point:

| | retry? | why |
| --- | --- | --- |
| 503 / 5xx / timeout | **yes** | the appliance is busy or broken; it may not be in a moment |
| 4xx | **no** | your request is malformed; it will be just as malformed next time |
| `success: false` | **no** | the request was fine and the answer is no |

A retry loop that retries everything will hammer an appliance for five attempts over a typo.

In [ ]:
disarm();

## 4. Truncated data — the nasty one

What happens if a connection drops mid-map? Guess first: you probably expect a short map, or an
exception.

It is worse than that.

In [ ]:
print(arm("truncate-map")["teaches"])

In [ ]:
truncated = httpx2.post(maprt.api_url + "/integration/GetMap", headers=headers,
                        json=body, timeout=120)

print("status      :", truncated.status_code)
print("has DONE    :", "DONE" in truncated.text)
print("last line   :", truncated.text.strip().splitlines()[-1])

No error. The status is 200 and the body is *plausible*. Now parse it:

In [ ]:
image, x_ticks, y_ticks = wire.parse_map_response(truncated.text)
print("parsed shape:", image.shape)

In [ ]:
disarm()
complete, _x, _y = wire.parse_map_response(
    httpx2.post(maprt.api_url + "/integration/GetMap", headers=headers,
                json=body, timeout=120).text)
print("complete shape:", complete.shape)

## Read those two shapes

```
truncated : (180, 185)
complete  : (360, 185)
```

**Half the map. No exception. Nothing in the data says anything is wrong.**

The parser works out the gantry axis from the values it actually sees. Half the triples means half
the gantry angles, so it builds a half-height array — perfectly consistent, entirely wrong.

What happens next is the dangerous part. Your code indexes by gantry row:

```python
image[row_for(200), column_for(0)]
```

and gets the data for **a completely different angle**, or an `IndexError` far from the cause. You
would be checking beam deliverability against a map that silently is not the map you asked for.

**This is what `DONE` is for.** Check it. Every time.

In [ ]:
def parse_map_safely(text, expect_gantry_rows=360):
    """Parse a GetMap body, refusing anything that did not arrive complete."""
    if not text.strip().endswith("DONE"):
        raise ValueError("the map response is truncated — no DONE terminator. "
                         "Do not use it: it will parse into a wrong-shaped array.")

    tokens = text.split()
    couch_samples, gantry_samples = (int(v) for v in tokens[0].split(","))
    expected = couch_samples * gantry_samples
    received = len(tokens) - 2                  # minus header, minus DONE
    if received != expected:
        raise ValueError(f"the map header promised {expected:,} triples but {received:,} arrived")

    image, x_ticks, y_ticks = wire.parse_map_response(text)
    if image.shape[0] != expect_gantry_rows:
        raise ValueError(f"the map parsed to {image.shape[0]} gantry rows, not "
                         f"{expect_gantry_rows}")
    return image, x_ticks, y_ticks


arm("truncate-map")
try:
    parse_map_safely(httpx2.post(maprt.api_url + "/integration/GetMap", headers=headers,
                                 json=body, timeout=120).text)
except ValueError as problem:
    print("caught:", problem)
disarm();

Three checks, any one of which would have caught it. The terminator is the cheapest.

## 5. Order is not guaranteed

§5.2.6 says *"the list is not ordered."* It means it. Each triple carries its own couch and gantry
angle precisely so that arrival position cannot matter.

In [ ]:
print(arm("shuffle-map")["teaches"])

In [ ]:
import numpy as np

shuffled = httpx2.post(maprt.api_url + "/integration/GetMap", headers=headers,
                       json=body, timeout=120).text
disarm()
ordered = httpx2.post(maprt.api_url + "/integration/GetMap", headers=headers,
                      json=body, timeout=120).text

print("the bytes differ      :", shuffled != ordered)
print("the first triple was  :", shuffled.splitlines()[1], "vs", ordered.splitlines()[1])
print("they parse identically:",
      bool(np.array_equal(wire.parse_map_response(shuffled)[0],
                          wire.parse_map_response(ordered)[0])))

Identical maps. Any code that did `triples[0]` expecting couch 90 / gantry 180 would be quietly
wrong here and nowhere else — the worst kind of bug to find.

## 6. A dead token

Tokens get rotated. Appliances get replaced.

In [ ]:
print(arm("bad-token")["teaches"])

In [ ]:
response = httpx2.get(maprt.api_url + "/integration/rooms", headers=headers)
print("status   :", response.status_code)
print("errorCode:", response.json()["errorCode"])
disarm();

Still a 200. Notebook 02's lesson, in the form you will actually meet it: a credential that worked
yesterday, failing today, with a status code that says everything is fine.

An `auth` category error means *check the configuration* — not *retry*, and not *the patient has
no surfaces*.

## Your turn

Write `robust_get_map(body)` that handles all of it: sensible timeout, retry on 503 and 5xx with
backoff, no retry on 4xx or `success: false`, and a truncation check.

Then prove it by arming `throttle` and calling it.

In [ ]:
# Your turn.
def robust_get_map(body):
    ...

<details><summary>Answer</summary>

Run the next cell.

</details>

In [ ]:
def robust_get_map(body, attempts=4, timeout=600.0):
    """Fetch a clearance map, surviving everything this notebook armed."""
    delay = 1.0
    for attempt in range(1, attempts + 1):
        try:
            response = httpx2.post(maprt.api_url + "/integration/GetMap",
                                   headers=headers, json=body, timeout=timeout)
        except httpx2.TimeoutException:
            if attempt == attempts:
                raise
            wait = delay
        else:
            if response.status_code == 503:
                wait = float(response.headers.get("Retry-After", delay))
            elif response.status_code >= 500:
                wait = delay
            elif response.status_code >= 400:
                response.raise_for_status()          # our bug; do not retry
            else:
                # 200 — but GetMap answers failure as JSON and success as text, so look.
                if response.text.lstrip().startswith("{"):
                    envelope = response.json()
                    raise MapRTApiError(envelope["errorCode"], envelope["errorCodeType"])
                return parse_map_safely(response.text)

        print(f"  attempt {attempt}: retrying in {wait:.0f}s")
        time.sleep(wait)
        delay = min(delay * 2, 30.0)

    raise RuntimeError("GetMap failed after every attempt")


arm("throttle")
image, x_ticks, y_ticks = robust_get_map(body)
print("got a map:", image.shape, f"{image.mean():.1%} clear")
disarm();

## Before you leave

**Always disarm.** A scenario left armed will confuse you an hour later in a different notebook.

In [ ]:
state = httpx2.get(SIM + "/state").json()
print("armed scenario:", state["scenario"]["name"])
assert state["scenario"]["name"] == "ok", "something is still armed — run disarm()"
print("clean.")

---

**Next:** [`08_capstone_plan_qa.ipynb`](08_capstone_plan_qa.ipynb) — all of it at once, on a real
RT Plan.